# CommerceFlow: Databricks medallion demonstration

Upload this project to a Databricks workspace Git folder. Use a compatible Spark 3.5 / Delta 3.2 runtime. This notebook demonstrates the same real UCI Bronze, Silver, quarantine and Gold transformations; it does not deploy Azure services. Put the original Excel dataset in a Unity Catalog Volume you can write to. Install `pandas` and `openpyxl` if your runtime does not include them. Set the widgets below before running. No payment credentials are required for the local application.

In [ ]:
dbutils.widgets.text('project_path', '')
dbutils.widgets.text('source_excel', '')
dbutils.widgets.text('output_volume', '')
dbutils.widgets.text('sample_rows', '50000')

In [ ]:
import sys, uuid
from pathlib import Path
project = dbutils.widgets.get('project_path')
source = dbutils.widgets.get('source_excel')
output = dbutils.widgets.get('output_volume')
if not project or not source or not output.startswith('/Volumes/'):
    raise ValueError('Set project_path, source_excel and a writable /Volumes/... output_volume')
sys.path.insert(0, project)
from pipelines.ingestion import ingest
from pipelines.validation import validate
from pipelines.transformation import gold_tables
spark.conf.set('spark.sql.caseSensitive', 'true')
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.conf.set('spark.sql.legacy.timeParserPolicy', 'CORRECTED')
run_id = uuid.uuid4().hex
run_path = Path(output) / 'commerceflow' / run_id
run_path.mkdir(parents=True, exist_ok=False)
sample = dbutils.widgets.get('sample_rows').strip()
bronze, source_hash = ingest(spark, Path(source), run_path, run_id, int(sample) if sample else None)
display(bronze.limit(20))

In [ ]:
silver, quarantine, stats = validate(bronze)
silver.write.format('delta').mode('overwrite').save(str(run_path / 'silver'))
quarantine.write.format('delta').mode('overwrite').save(str(run_path / 'quarantine'))
assert bronze.count() == silver.count() + quarantine.count()
print(stats)
display(quarantine.limit(20))

In [ ]:
tables = gold_tables(spark, silver)
for name, frame in tables.items():
    frame.write.format('delta').mode('overwrite').save(str(run_path / 'gold' / name))
assert tables['fact_transactions'].count() == silver.count()
fact_total = tables['fact_transactions'].selectExpr('sum(gross_amount-return_amount)').first()[0]
for name in ['agg_daily_sales', 'agg_country_sales', 'agg_product_sales']:
    assert tables[name].selectExpr('sum(net_sales)').first()[0] == fact_total
display(tables['agg_daily_sales'].orderBy('date'))

## Proposed migration mapping

| Local application | Proposed Azure equivalent |
|---|---|
| Local Delta storage | ADLS Gen2 |
| Local Spark | Azure Databricks |
| Python subprocess orchestration | Azure Data Factory |
| DuckDB | Cloud analytical warehouse |
| Streamlit | BI layer |

This mapping is architectural guidance, not a deployed Azure solution. Databricks Free Edition availability and supported runtime features depend on the account. Data source: Daqing Chen (2015), [Online Retail, UCI](https://doi.org/10.24432/C5BW33), CC BY 4.0.